# ISP - EOT Submission

## Task 1.1 - Vehicle Detection & Tracking

### Overview
This cell defines and runs the `TrafficDetector` class, which detects and tracks moving vehicles in a traffic video using **background subtraction** and **contour detection**.

### How Pipeline Works?
1. **Open video** - loads the video file and validates it
2. **Crop to Region of Interest (IR)** - focuses only on the main street area
3. **Gaussian Blur** - smooths the frame to reduce noise before detection
4. **KNN Background Subtraction** - separates moving foreground (cars) from static background
5. **Thresholding** - converts the mask to clean black & white
6. **Morphological Closing** - fills small holes and gaps in detected car shapes
7. **Contour Detection** - finds outlines of detected objects
8. **Area Filtering** - removes objects too small to be cars
9. **Bounding Boxes** - draws green rectangles around detected vehicles
10. **IR Boundary** - draws red rectangle showing the monitored region

### Key Parameters
- `min_area = 4000` - minimum contour area to be considered a vehicle
- `history = 10000` - number of past frames used to model the background
- `dist2Threshold = 900` - sensitivity of KNN detection (lower = more sensitive)
- `IR_TOP/BOTTOM = 265, 590` - vertical crop of the region of interest
- `IR_LEFT/RIGHT = 0, 1040` - horizontal crop of the region of interest

### Video Controls
- Press **`q`** to stop the video at any time

## Install Necessary Packages

In [1]:
!pip install numpy
!pip install opencv-python

## Import Libraries

In [2]:
import numpy as np
import cv2
import os
base_directory = os.getcwd()
print(base_directory)

C:\Users\Aswat\Exercise_1_ISP_Final


## Details Of Video File & Interested Region Analysis
Before running the full detection, I first need to verify that the video loads correctly and that the interested region is covering the right area of the frame.

In [3]:
path_of_vid_file = os.path.join(base_directory, "Exercise1_Files", "Traffic_Laramie_1.mp4")
path_to_save_info = os.path.join(base_directory)

vid_file = cv2.VideoCapture(path_of_vid_file)

# Info of video file
print("~~~ Properties Of Video: ~~~")
print(f"Opened: {vid_file.isOpened()}")
print(f"Width:  {int(vid_file.get(cv2.CAP_PROP_FRAME_WIDTH))}")
print(f"Height: {int(vid_file.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
print(f"FPS:    {vid_file.get(cv2.CAP_PROP_FPS)}")
print(f"Total frames: {int(vid_file.get(cv2.CAP_PROP_FRAME_COUNT))}")

# To test the interested region we read the first few frames - IR stands for interested region
ret, frame = vid_file.read()
if ret:
    print(f"\n~~~ FRAME 0 SIZE: {frame.shape} (height, width, channels) ~~~")
    
    # Test interested region coordinates
    # y start - y end to slice height
    IR_TOP, IR_BOTTOM = 265, 590
    # x start - x end to slice width
    IR_LEFT, IR_RIGHT = 0, 1040

    # IR test
    print(f"\n~~~ IR TEST ~~~")
    print(f"IR coords: y = {IR_TOP}:{IR_BOTTOM}, x = {IR_LEFT}:{IR_RIGHT}")
    print(f"IR size:   {(IR_BOTTOM - IR_TOP)} x {(IR_RIGHT - IR_LEFT)}")
    
    IR = frame[IR_TOP:IR_BOTTOM, IR_LEFT:IR_RIGHT]
    print(f"Actual IR shape: {IR.shape}")
    
    # Save IR image to inspect visually
    save_image = os.path.join(path_to_save_info, "IR_test.jpg")
    cv2.imwrite(save_image, IR)
    print("IR_test.jpg Saved Successfully!")
    
    # Save full frame with IR box
    output = frame.copy()
    cv2.rectangle(output, (IR_LEFT, IR_TOP), (IR_RIGHT, IR_BOTTOM), (0, 0, 255), 3)
    save_frame = os.path.join(path_to_save_info, "frame_with_IR.jpg")
    cv2.imwrite(save_frame, output)
    print("frame_with_IR.jpg Saved Successfully!")
else:
    print("Cannot read first frame. Please check video path!")
    
vid_file.release()

~~~ Properties Of Video: ~~~
Opened: True
Width:  1040
Height: 600
FPS:    25.0
Total frames: 4448

~~~ FRAME 0 SIZE: (600, 1040, 3) (height, width, channels) ~~~

~~~ IR TEST ~~~
IR coords: y = 265:590, x = 0:1040
IR size:   325 x 1040
Actual IR shape: (325, 1040, 3)
IR_test.jpg Saved Successfully!
frame_with_IR.jpg Saved Successfully!


## Vehicle Detection & Tracking Using Background Subtraction

In [4]:
class TrafficDetector:

    def __init__(self, video_path, min_area=4000):
        # Store path to the video file
        self.video_path    = video_path
        # Minimum contour area which helps to filters out small noise & non-vehicle objects
        self.min_area      = min_area
        self.video_capture = None

        # KNN background subtractor which separates moving cars from static background
        # history: number of past frames used to build background model
        # dist2Threshold: squared distance threshold for pixel classification
        # detectShadows: disabled to avoid shadow pixels being treated as foreground
        self.background_remover = cv2.createBackgroundSubtractorKNN(detectShadows=False, history=10000, dist2Threshold=900)

        self.kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

        # Interested region
        self.IR_TOP, self.IR_BOTTOM = 265, 590
        self.IR_LEFT, self.IR_RIGHT = 0, 1040

    def get_foreground_mask(self, frame):

        # Crop interested region
        ir_frame = frame[self.IR_TOP:self.IR_BOTTOM, self.IR_LEFT:self.IR_RIGHT]

        # Apply Gaussian blur to reduce noise before background subtraction
        smoothed = cv2.GaussianBlur(ir_frame, (5, 5), 0)

        # Apply KNN background subtraction which outputs a mask of cars in white foreground
        mask = self.background_remover.apply(smoothed)

        # Threshold the mask to strict black & white removing any grey shadow pixels
        _, mask = cv2.threshold(mask, 100, 255, cv2.THRESH_BINARY)

        # Morphological closing that fills small holes and gaps within detected car shapes
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, self.kernel, iterations=2)

        return mask

    def get_bounding_boxes(self, mask):

        # Find external contours of all white regions in the mask
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Filter out contours too small to be vehicles
        contours = [c for c in contours if cv2.contourArea(c) > self.min_area]

        bounding_boxes = []
        for contour in contours:
            # Get bounding rectangle for each contour
            x, y, w, h = cv2.boundingRect(contour)

            # Offset coordinates back to full frame position 
            bounding_boxes.append((
                x + self.IR_LEFT,
                y + self.IR_TOP,
                x + self.IR_LEFT + w,
                y + self.IR_TOP + h
            ))
        return bounding_boxes

    def draw_boxes(self, frame, bounding_boxes):

        # Draw a green rectangle around each detected vehicle
        for (x, y, x2, y2) in bounding_boxes:
            cv2.rectangle(frame, (x, y), (x2, y2), (0, 255, 0), 2)

        # Draw red rectangle showing the monitored IR boundary
        cv2.rectangle(frame,
                      (self.IR_LEFT, self.IR_TOP),
                      (self.IR_RIGHT, self.IR_BOTTOM),
                      (0, 0, 255), 2)

    def open_video(self):

        # Load the video file from the given path
        self.video_capture = cv2.VideoCapture(self.video_path)

        # Check if the video opened successfully else print error and return False if not
        if not self.video_capture.isOpened():
            print(f"Error: Could not open video: {self.video_path}")
            return False
        return True

    def run(self):

        # Open video and exit if it fails
        if not self.open_video():
            return

        # Get the video frame rate to sync playback speed
        fps = self.video_capture.get(cv2.CAP_PROP_FPS)

        # Calculate wait time per frame in milliseconds
        wait_ms = int(1000 / (fps * 3))

        while True:
            # Read the next frame from the video
            ret, frame = self.video_capture.read()

            # Stop loop if no more frames are available
            if not ret:
                break

            # Generate the foreground mask from the current frame
            mask = self.get_foreground_mask(frame)

            # Get bounding boxes of detected vehicles
            bounding_boxes = self.get_bounding_boxes(mask)

            # Draw boxes and IR boundary on the frame
            self.draw_boxes(frame, bounding_boxes)

            # Display the annotated frame in the Live Video window
            cv2.imshow("Live Video", frame)

            # Exit the loop if the user presses 'q'
            if cv2.waitKey(wait_ms) & 0xFF == ord('q'):
                break

        # Release video resources and close all display windows
        self.video_capture.release()
        cv2.destroyAllWindows()


# Run code section
path_of_vid_file = os.path.join(base_directory, "Exercise1_Files", "Traffic_Laramie_1.mp4")
detector = TrafficDetector(video_path=path_of_vid_file, min_area=4000)
detector.run()